In [1]:
import os
from typing import Dict, List

import duckdb
import numpy as np
import polars as pl
from dotenv import load_dotenv
from Levenshtein import ratio
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

open_router_key = os.getenv("OPEN_ROUTER_API_KEY")

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=open_router_key,
)
con = duckdb.connect("tesco_raw.db")
# rows = con.execute("DESCRIBE tesco_products_embeddings").fetchall()
# schema_str = "\n".join(f"- {name} ({dtype})" for name, dtype, *_ in rows)
# con.install_extension("vss", repository="core")
# con.load_extension("vss")
# con.sql("CREATE OR REPLACE SECRET secret (TYPE s3,PROVIDER credential_chain);")

In [2]:
def format_as_markdown_kv(data: List[Dict]) -> str:
    """Convert data to markdown with fenced key-value blocks."""
    lines = ["# Supermarket Product Database\n"]

    for i,record in enumerate(data):
        lines.append(f"## Record {i+1}\n")
        lines.append("```")
        for key, value in record.items():
            lines.append(f"{key}: {value}")
        lines.append("```\n")

    return "\n".join(lines)

In [ ]:
# Create raw SQL table with FTS BM25 index
con.sql("""
    CREATE OR REPLACE TABLE tesco_products AS
    (SELECT DISTINCT
        tpnc,
        data.title AS title,
        data.brandName AS brand,
        data.details.packSize[1].value AS quantity,
        data.details.packSize[1].units AS unit,
        data.price.actual AS price,
        data.promotions[1].startDate AS promotion_start_date,
        data.promotions[1].endDate AS promotion_end_date,
        data.price.unitPrice AS unit_price,
        data.price.unitOfMeasure AS unit_of_measure,
        data.promotions[1].metaData.seo.afterDiscountPrice AS clubcard_price,
        data.superDepartmentName AS department,
        data.departmentName AS sub_department,
        data.aisleName AS aisle,
        data.shelfName AS shelf,
        data.description AS description,
        data.gtin AS gtin,
        CURRENT_TIMESTAMP AS scraped_at
    FROM 's3://ie-supermarket-data/raw/tesco/2026-02-04/*.jsonl.gz'
    WHERE data IS NOT NULL)
""")

con.sql(
    "INSTALL fts;\
            LOAD fts;\
            PRAGMA create_fts_index('tesco_products', 'tpnc', 'title', overwrite=1);"
)

In [3]:
df = con.sql("SELECT * FROM tesco_products_embeddings").pl()
shelf_embeddings_dict = con.sql("SELECT * FROM tesco_shelf_embeddings").pl().to_dict(as_series=False)

In [4]:
aisle_to_shelf_mapping = df.group_by(pl.col('aisle')).agg(pl.col('shelf').unique().alias('shelves'), pl.col('shelf').n_unique().alias('count_shelves')).sort(by = 'count_shelves', descending= True)
# .order_by(pl.col('count_shelves').desc())

In [7]:
aisle_to_shelf_mapping.filter(pl.col('aisle') == "White Wine")['shelves'].to_list()

[['Sauvignon Blanc',
  'Italy',
  'Spain',
  'Chardonnay',
  'Small Bottles',
  'Chile',
  'Australia',
  'Other',
  'South Africa',
  'France',
  'Argentina',
  'North America',
  'New Zealand',
  'Pinot Grigio',
  'Germany',
  'California']]

In [10]:
# After generating embeddings
# shelf_centroids = {}

# for category in df['shelf'].drop_nulls().unique().to_list():
#     mask = df["shelf"] == category
#     cat_embeddings = np.vstack(df['embeddings'].filter(mask))
#     centroid = cat_embeddings.mean(axis=0).reshape(1, -1)
#     shelf_centroids[f'{category}'] = centroid

# centroids_v = np.vstack(list(shelf_centroids.values()))
# dict_keys=list(shelf_centroids.keys())


shelf_centroids_v = np.vstack(shelf_embeddings_dict['centroid_embeddings'])
dict_keys= shelf_embeddings_dict['shelf_name']
 #test one product
test_product = np.vstack(df.filter(pl.col('tpnc')=='292769876').select('embeddings'))
sim = cosine_similarity(shelf_centroids_v, test_product).flatten()
ind = np.argpartition(sim,kth= -10)[-10:]
top5 = ind[np.argsort(sim[ind])[::-1]]

for i in top5:
    print(f'Shelf:{dict_keys[i]}')
    print(f'Similarity: {sim[i]}\n')

Shelf:Whole Milk
Similarity: 0.9320906247782109

Shelf:Protein Milk
Similarity: 0.7941993641913713

Shelf:Slimline Milk
Similarity: 0.7650354358496375

Shelf:Organic Milk
Similarity: 0.7638931100011448

Shelf:Low fat milk
Similarity: 0.7590205083458657

Shelf:Milkshakes
Similarity: 0.7434412118128177

Shelf:Chocolate Milkshake
Similarity: 0.7309968567229082

Shelf:Skimmed Milk
Similarity: 0.7238972869727213

Shelf:Condensed Milk
Similarity: 0.7182085652533088

Shelf:Butter Milk
Similarity: 0.7150559078391047



In [11]:
# Similarities between different shelves
shelf_similarities = cosine_similarity(shelf_centroids_v)
key_index = dict_keys.index('Sauvignon Blanc')
shelf_test = shelf_similarities[key_index]
# ind = np.argpartition(shelf_test ,kth= -5)[-5:]
top5 = np.argsort(shelf_test)[::-1][:20]
print(f"Reference Shelf: {dict_keys[key_index]}\n")
for i in top5:
    print(f'Shelf:{dict_keys[i]}')
    print(f'Similarity: {shelf_test[i]}\n')

Reference Shelf: Sauvignon Blanc

Shelf:Sauvignon Blanc
Similarity: 1.0000000000000009

Shelf:Cabernet Sauvignon
Similarity: 0.8780423152060957

Shelf:No Alcohol
Similarity: 0.8480371312930994

Shelf:White Wine
Similarity: 0.8250196869627686

Shelf:New Zealand
Similarity: 0.8156602064267481

Shelf:Chile
Similarity: 0.7808376557993051

Shelf:Finest Red Wine
Similarity: 0.7793724336257917

Shelf:Champagne
Similarity: 0.7783610480615631

Shelf:South Africa
Similarity: 0.7713846242754299

Shelf:Red Wine
Similarity: 0.7654348722959249

Shelf:France
Similarity: 0.7601961157479152

Shelf:No & Low Alcohol White Wine
Similarity: 0.7598657307394401

Shelf:Champagne & Sparkling Wine
Similarity: 0.7517438708569141

Shelf:Australia
Similarity: 0.7419187912598435

Shelf:Chardonnay
Similarity: 0.7264567636648191

Shelf:Sparkling Wine
Similarity: 0.7197092458926257

Shelf:Small Bottles
Similarity: 0.7186212122470907

Shelf:Pinot Noir
Similarity: 0.717556666076566

Shelf:Rose Wine
Similarity: 0.7030409

In [ ]:
# Adding nested struct to polars df, maybe not most practical will need to revisit. Want to have a record of what categories and sub-categories each product can belong to though
similar_categories = [[
    {"Shelf": "Frozen Joints", "Similarity": 0.8850135803222656},
    {"Shelf": "Turkey Joints & Thighs", "Similarity": 0.6926565170288086},
    {"Shelf": "Frozen Chicken Goujonsy", "Similarity": 0.6674225926399231},
    {"Shelf": "Frozen Classic Meals", "Similarity": 0.6656007766723633},
    {"Shelf": "Frozen Shredded Chicken", "Similarity": 0.639082670211792},
]]

df.filter(pl.col('tpnc') == '272866077').with_columns(
    pl.Series("similar_categories", similar_categories)
)

In [266]:
brands_list  = df['brand'].unique().to_list()
brands_list

['MARQUES DE RISCA',
 'MIXX AUDIO',
 'Jammie DODGER',
 'Princess',
 'LYCLEAR',
 'MINERVA',
 'DR MOO QUICK MILK',
 'ASKEYS',
 'BISODOL',
 'SCOTCH',
 'febreze',
 'RHYTHM',
 'WASH & GO',
 'BATCHELORS',
 'EUROPASTRY',
 'PUREVIA',
 'HEAT & EAT',
 'Dr. Oetker',
 'PERNOD',
 "Joey's",
 'got2b',
 'CALGON',
 'WHISPERING ANGEL',
 'LOR',
 'EAGLE HAWK',
 'TASSIMO',
 'MARTENS',
 'RADOX',
 'MRS BALLS',
 'TERRE DUCALI',
 'FUSCO',
 'JUNIOR GAME',
 'joie',
 'LA ROSE BOUQUEY',
 'ROBINSON YOUNG',
 'TS FOODS',
 'YAZOO',
 'DOTS',
 'CLOUDY BAY',
 'SPRITE',
 'CAMPARI',
 'VILEDA',
 'AIR WICK',
 'SAFE AND SOUND',
 'VILLA MARIA',
 'BRADY FAMILY',
 'OTE',
 'LAURENT MIQUEL',
 'Disney',
 'TASTE OF GOODNESS',
 'LONDON ESSENCE',
 'The Laughing Cow',
 'SPARKLING',
 'CHIVAS REGAL',
 'Funky Monkey',
 'NEAT',
 'ALBERTO BALSAM',
 'COFFEE MATE',
 'Gallo',
 'WALKERS',
 'DAILY MIRROR',
 'CARLSBERG',
 'Heverlee',
 'Paperchase',
 'CHEETOS',
 'BUTTERKRUST',
 'R ROBERTS',
 'MIGHTY MUSHROOM',
 'LYLES',
 'YVONNE ELLEN',
 'BAY FISH

In [ ]:
brands_list  = df['brand'].unique().to_list()

def get_brand_if_misspelled(brand:str , brands_list: List[str]) -> str:
    lev_similarities = [ratio(x.lower(),brand, score_cutoff=0.4) for x in brands_list]
    max_index = lev_similarities.index(max(lev_similarities))
    return brands_list[max_index]

brand = 'cadbry'
if brand not in (brands_list):
    brand = get_brand_if_misspelled(brand,brands_list)
else:
    pass

In [ ]:
(3782/1000000)*0.20*1000000

75.64

In [ ]:
system_prompt = """You are a DuckDB SQL expert. Your task is to convert natural language queries about Irish supermarket products into executable DuckDB SQL statements.

## Database Schema

You will be working with a DuckDB table called `tesco_products_embeddings`. Here is the complete schema:

**Table: tesco_products_embeddings**

Columns:
- tpnc (VARCHAR) - Product code
- title (VARCHAR) - Product title
- brand (VARCHAR) - Brand name
- quantity (VARCHAR) - Quantity/size
- unit (VARCHAR) - Unit type
- price (DOUBLE) - Regular price
- promotion_start_date (TIMESTAMP) - Start of promotion period
- promotion_end_date (TIMESTAMP) - End of promotion period
- unit_price (DOUBLE) - Price per unit
- unit_of_measure (VARCHAR) - Unit of measurement
- clubcard_price (DOUBLE) - Clubcard discount price
- department (VARCHAR) - Top-level category
- sub_department (VARCHAR) - Second-level category
- aisle (VARCHAR) - Aisle location/category
- shelf (VARCHAR) - Shelf location/subcategory
- description (VARCHAR) - Product description
- gtin (VARCHAR) - Global Trade Item Number
- scraped_at (TIMESTAMP WITH TIME ZONE) - Data collection timestamp
- product_text (VARCHAR) - Full-text searchable field
- embeddings (FLOAT[768]) - Vector embeddings

**Full-Text Search Function:**
The table has a full-text search index accessible via `fts_main_tesco_products.match_bm25()`. Use this function when searching by product name, description, or text-based characteristics. This function returns a relevance score that can be used for ordering results.

## User Query Input

The user's query has been pre-processed and includes both a natural language query and extracted metadata. Here is the query with its metadata:

The metadata may include:
- **product_name**: The identified product name from the query
- **product_type**: The type/category of product
- **brand**: Specific brand if mentioned
- **supermarket**: The supermarket chain (Tesco in this case)
- **quantity**: Specific quantity or size if mentioned
- **aisles**: A list of relevant aisles for this product
- **shelves**: A list of relevant shelf subcategories within those aisles

## Your Task

Generate a valid DuckDB SQL query that retrieves the products the user is looking for based on their natural language query and the provided metadata.

## Critical Requirements

### 1. Mandatory Filters
You MUST include filters for both `aisle` AND `shelf` in your WHERE clause. These are required filters that must always be present.

### 2. Shelf Selection Logic - IMPORTANT
When multiple shelves are provided in the metadata, you must evaluate each shelf to determine whether to include it in your query. **Your approach should be INCLUSIVE rather than EXCLUSIVE.**

**Key principle:** Include ANY shelf that could potentially contain products matching the user's search terms. When in doubt, include the shelf and let the BM25 relevance scoring rank the results appropriately.

**Guidance:**
- If the user searches for a specific product (e.g., "sauvignon blanc"), include ALL shelves that could contain that product, even if the shelf name seems broad (e.g., country names like "Italy", "Spain", "Chile")
- The shelf name doesn't have to be an exact match to the search term - if the shelf could logically contain products matching the search, include it
- For example, if searching for "sauvignon blanc" and shelves include "Sauvignon Blanc", "Italy", "Spain", "Chile", "Australia", "Chardonnay", include all EXCEPT "Chardonnay" because only Chardonnay is a different wine type that definitely won't contain sauvignon blanc
- Rely on BM25 full-text search scoring to rank the most relevant products at the top
- Only exclude shelves that clearly contain completely different products than what the user is asking for

### 3. Use Available Metadata
If any metadata fields are populated (brand, product_type, quantity, etc.), incorporate them as additional filters in your WHERE clause to narrow down results appropriately.

### 4. Full-Text Search
When searching by product names or text-based descriptions, use the `fts_main_tesco_products.match_bm25()` function. This enables relevance-based ranking of results.

### 5. Ordering Results
When using full-text search with BM25, always ORDER BY the BM25 score in DESCENDING order. This ensures the most relevant products appear first.

### 6. Common-Sense Assumptions
Apply reasonable defaults when the query is general:
- "milk" means regular dairy milk unless alternatives are specified
- "cheapest" means compare across all brands unless a specific brand is mentioned
- Apply similar common-sense interpretations for other product categories

## Analysis Process

Before writing your SQL query, work through the following steps inside <analysis> tags:

1. **Quote the natural language query**: Write out the exact user query in quotes to keep it clearly in focus.

2. **List the metadata**: Write out all metadata fields and their values from the user query. If a field is not provided, note it as "not provided".

3. **Extract search terms**: From the natural language query, identify and list the key words or phrases that should be used for searching products.

4. **Evaluate and select shelves**: If multiple shelves are provided, systematically evaluate each shelf:
   - For each shelf, determine whether it COULD contain products matching the user's search terms
   - Remember: Be INCLUSIVE. Include any shelf that might contain relevant products
   - Ask yourself: "Could products matching my search terms logically be found on this shelf?"
   - Only exclude shelves that clearly contain completely different products
   - For each shelf, write an explicit decision: "INCLUDE" or "EXCLUDE" followed by your justification
   - Provide a final list of all included shelves
   - It's OK for this section to be quite long if there are many shelves to evaluate.

5. **Determine filters and write SQL fragments**: Build your filter strategy step by step by writing out the actual SQL conditions:
   - Start with the mandatory aisle filter(s) - write the actual SQL: `aisle = 'value'` or `aisle IN ('value1', 'value2')`
   - Add the mandatory shelf filter(s) based on your evaluation above - write the actual SQL: `shelf = 'value'` or `shelf IN ('value1', 'value2', ...)`
   - Add any additional metadata filters (brand, product_type, quantity, etc.) if provided - write the actual SQL for each
   - Add any other conditions derived from the natural language query - write the actual SQL
   - If using BM25, write out the BM25 condition

6. **Verify mandatory filters**: Explicitly confirm that your WHERE clause includes BOTH an aisle filter AND a shelf filter.

7. **Plan the search strategy**: 
   - Decide whether to use BM25 full-text search (typically yes for text-based product searches)
   - If using BM25, specify what search terms you'll use in the match_bm25() function

8. **Build SQL components**:
   - SELECT clause: list which columns to return (if using BM25, include the relevance score)
   - FROM clause: the table name
   - WHERE clause: list each filter condition (these should be the SQL fragments you wrote in step 5)
   - ORDER BY clause: specify sorting logic (must include BM25 score DESC if using full-text search)
   - LIMIT clause: if appropriate based on the query

## Output Format

Your response must follow this structure:

<analysis>
1. Natural language query:
   "[exact user query]"

2. Metadata provided:
   - product_name: [value or "not provided"]
   - product_type: [value or "not provided"]
   - brand: [value or "not provided"]
   - quantity: [value or "not provided"]
   - aisles: [list or "not provided"]
   - shelves: [list or "not provided"]

3. Key search terms from query:
   - [term 1]
   - [term 2]
   - [etc.]

4. Shelf selection:
   - Shelf "[name]": INCLUDE/EXCLUDE - [justification focusing on whether products matching search terms COULD be on this shelf]
   - Shelf "[name]": INCLUDE/EXCLUDE - [justification]
   - [repeat for each shelf]
   - Final selection: [list of all included shelves]

5. Filter strategy (SQL fragments):
   - Mandatory aisle filter: [actual SQL condition]
   - Mandatory shelf filter: [actual SQL condition]
   - [Additional filter 1]: [actual SQL condition]
   - [Additional filter 2]: [actual SQL condition]
   - [etc.]

6. Verification:
   - Aisle filter present: [yes/no]
   - Shelf filter present: [yes/no]

7. Search strategy:
   - Using BM25: [yes/no]
   - Search terms: [terms to search for, if using BM25]
   - BM25 SQL fragment: [actual SQL condition if using BM25]

8. SQL components:
   - SELECT: [columns, including relevance score if using BM25]
   - FROM: [table]
   - WHERE:
     - [condition 1]
     - [condition 2]
     - [condition 3]
     - [etc.]
   - ORDER BY: [sorting]
   - LIMIT: [number or "not applicable"]
</analysis>

<sql>
[Your complete, executable DuckDB SQL statement]
</sql>

Example structure:

<analysis>
1. Natural language query:
   "find organic apples under 3 euros"

2. Metadata provided:
   - product_name: "apples"
   - product_type: "fruit"
   - brand: not provided
   - quantity: not provided
   - aisles: ["Fresh Produce"]
   - shelves: ["Apples", "Organic Fruit"]

3. Key search terms from query:
   - organic
   - apples

4. Shelf selection:
   - Shelf "Apples": INCLUDE - directly contains apples, could have organic varieties
   - Shelf "Organic Fruit": INCLUDE - contains organic products, could have apples
   - Final selection: ["Apples", "Organic Fruit"]

5. Filter strategy (SQL fragments):
   - Mandatory aisle filter: aisle = 'Fresh Produce'
   - Mandatory shelf filter: shelf IN ('Apples', 'Organic Fruit')
   - Price filter: price < 3.0
   - BM25 filter: fts_main_tesco_products.match_bm25(tpnc, 'organic apples') IS NOT NULL

6. Verification:
   - Aisle filter present: yes
   - Shelf filter present: yes

7. Search strategy:
   - Using BM25: yes
   - Search terms: "organic apples"
   - BM25 SQL fragment: fts_main_tesco_products.match_bm25(tpnc, 'organic apples')

8. SQL components:
   - SELECT: title, brand, price, shelf, fts_main_tesco_products.match_bm25(tpnc, 'organic apples') as relevance_score
   - FROM: tesco_products_embeddings
   - WHERE:
     - aisle = 'Fresh Produce'
     - shelf IN ('Apples', 'Organic Fruit')
     - price < 3.0
     - fts_main_tesco_products.match_bm25(tpnc, 'organic apples') IS NOT NULL
   - ORDER BY: relevance_score DESC
   - LIMIT: 20
</analysis>

<sql>
SELECT
    title,
    brand,
    price,
    shelf,
    fts_main_tesco_products.match_bm25(tpnc, 'organic apples') as relevance_score
FROM tesco_products_embeddings
WHERE
    aisle = 'Fresh Produce'
    AND shelf IN ('Apples', 'Organic Fruit')
    AND price < 3.0
    AND fts_main_tesco_products.match_bm25(tpnc, 'organic apples') IS NOT NULL
ORDER BY relevance_score DESC
LIMIT 20;
</sql>

Now, please analyze the user query provided above and generate the appropriate SQL query.

Return ONLY the raw SQL statement. No explanation, no markdown, no tags."""

In [ ]:
def build_user_input(query: str, metadata: dict) -> str:
    return f"""Natural language query: {query}

Metadata:
- product_name: {metadata.get('product_name', 'not provided')}
- product_type: {metadata.get('product_type', 'not provided')}
- brand: {metadata.get('brand', 'not provided')}
- quantity: {metadata.get('quantity', 'not provided')}
- aisles: {metadata.get('aisles', 'not provided')}
- shelves: {metadata.get('shelves', 'not provided')}"""

user_input = build_user_input(
    query="what is the average price of a 750 ml bottle of wine",
    metadata={
        "product_name": "sauvignon blanc",
        "brand": None,
        "aisles": ["White Wine"],
        "shelves": ['Sauvignon Blanc','Italy','Spain','Chardonnay','Small Bottles','Chile','Australia','Other','South Africa','France','Argentina','North America','New Zealand','Pinot Grigio','Germany','California'],
    }
)

In [60]:
# Generate SQL query
import openai
import mlflow

# Enable auto-tracing for OpenAI
mlflow.openai.autolog()

# Set a tracking URI and an experiment
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("supermarket_token_tracking")

response = client.responses.create(
    model="deepseek/deepseek-chat-v3.1",
    instructions= system_prompt,
    input=user_input,
)


Trace(trace_id=tr-30c2436a3c4282439867e22b13a835e3)

In [82]:
con.sql("""SELECT aisle, AVG(match_score)
        FROM(SELECT aisle , fts_main_tesco_products.match_bm25(tpnc, 'wine') as match_score, embeddings as embedding
        FROM tesco_products_embeddings
        WHERE match_score IS NOT NULL)
        GROUP BY aisle
        ORDER BY AVG(match_score) DESC""")

┌─────────────────────────────────────────┬────────────────────┐
│                  aisle                  │  avg(match_score)  │
│                 varchar                 │       double       │
├─────────────────────────────────────────┼────────────────────┤
│ Sweets                                  │  2.899469380251816 │
│ Jelly & Chewy Sweets                    │  2.899469380251816 │
│ Traditional Sweets & Confectionery      │  2.656996267687693 │
│ Rose                                    │  2.451947907773221 │
│ Japanese                                │  2.451947907773221 │
│ Packet Mix & Traditional Sauces         │  2.451947907773221 │
│ Stock Cubes & Pots                      │  2.393392112586822 │
│ Vinegar                                 │ 2.3641142149936223 │
│ Red Wine                                │   2.33369921275188 │
│ Glassware                               │ 2.3077836273994174 │
│ White Wine                              │  2.277527395904224 │
│ No & Low Alcohol Wine  

In [22]:
results = con.sql("""SELECT title, brand, quantity, price, unit_price, clubcard_price, fts_main_tesco_products.match_bm25(tpnc, 'sauvignon blanc') as bm25_score
FROM tesco_products_embeddings
WHERE aisle = 'White Wine'
  AND shelf IN ('Sauvignon Blanc', 'Italy', 'Spain', 'Chile', 'Australia', 'South Africa', 'France', 'Argentina', 'North America', 'New Zealand')
  AND fts_main_tesco_products.match_bm25(tpnc, 'sauvignon blanc') > 0
ORDER BY bm25_score DESC;""").fetchdf()
results_str = results.to_json(orient="records", indent=2)
results_markdown_kv = format_as_markdown_kv(results.to_dict('records'))

In [41]:
# instructions="You summarise Irish supermarket product query results in a concise, helpful way. When comparing product prices pay particular attention to unit prices vs actual price and explore both options if not specified by the user. Base your answer on the actual data, not assumptions.",
stream = client.responses.create(
    model="openai/gpt-5-nano",
    instructions="You summarise Irish supermarket product query results in a concise, helpful way." \
    "When comparing product prices pay particular attention to unit prices vs actual price and explore both options if not specified by the user. Base your answer on the actual data, not assumptions. " \
    "When comparing value always use the unit price",
    input=f"User asked: what are the different sauvignon blancs available? \n\nQuery results:\n{results_markdown_kv}",
    stream =True
)
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

Here are the Sauvignon Blanc options from the results, with price, size, unit price (price per litre), and clubcard price if shown.

- JP CHENET Sauvignon Blanc 75Cl — £10.00; 0.75L; unit £13.33; clubcard £9.00
- Yealands Sauvignon Blanc 75Cl — £15.00; 0.75L; unit £20.00; clubcard £12.00
- Freixenet Sauvignon Blanc 750Ml — £15.00; 0.75L; unit £20.00; clubcard £10.00
- TRACES Sauvignon Blanc 75cl — £15.00; 0.75L; unit £20.00; clubcard £10.00
- Ah, Sauvignon Blanc 750ml — £15.00; 0.75L; unit £20.00; clubcard not listed
- MOST WANTED Sauvignon Blanc Can 187Ml — £3.00; 0.187L; unit £16.04; clubcard not listed
- MOST WANTED Sauvignon Blanc 75Cl — £13.50; 0.75L; unit £18.00; clubcard £12.00
- Bend In The River The Sauvignon Blanc 75Cl — £8.00; 0.75L; unit £10.67; clubcard not listed
- MIRLA BAY Sauvignon Blanc 750 Ml — £19.00; 0.75L; unit £25.33; clubcard not listed
- LIONSGATE Sauvignon-Blanc 75Cl — £15.00; 0.75L; unit £20.00; clubcard £7.50
- CALVET Prestige Sauvignon Blanc 75Cl — £20.00; 